# PennyLane Hybrid Jobs

Train a full PennyLane variational loop locally, then package it as a managed Braket Hybrid Job.

**Objectives:**
- Define a small PennyLane QNode on `default.qubit` and train it with a PennyLane optimizer on a tiny dataset
- Log the cost every step and plot the convergence curve from a loop that runs for real on your laptop
- Read the job entry-point that re-points the QNode at `braket.aws.qubit`, streams metrics with `log_metric`, and persists results with `save_job_result`
- Retrieve the optimal parameters at the end of training
- Keep every executed cell credential-free: the AWS submission stays gated behind `RUN_ON_AWS`

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp  # autograd-aware numpy for differentiable params

# Braket pieces -- importing is free and needs no credentials.
from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator
from braket.aws import AwsQuantumJob
from braket.jobs.metrics import log_metric
from braket.jobs import save_job_checkpoint, load_job_checkpoint, save_job_result

# Gate every AWS call behind this flag. Nothing below contacts AWS while it is False.
RUN_ON_AWS = False

np.random.seed(7)  # deterministic dataset + parameter init
print("PennyLane", qml.__version__, "| RUN_ON_AWS =", RUN_ON_AWS)

## 1. A tiny training set

A hybrid job earns its keep on *iteration*, so the first thing we need is something to iterate on. We will fit a one-feature regression target -- $f(x) = \sin(x)$ sampled on a small grid. Eight points is plenty to make the optimizer work without being slow, and the target is smooth enough that a two-qubit variational circuit can learn it. Everything here is plain numpy; no quantum hardware is touched yet.

In [ ]:
# Eight sample points on [0, 2pi]; targets are sin(x).
# requires_grad=False marks these as fixed data, not parameters to optimize.
X = pnp.linspace(0.0, 2 * np.pi, 8, requires_grad=False)
Y = pnp.array(np.sin(np.array(X)), requires_grad=False)

print("X:", np.round(np.array(X), 3))
print("Y:", np.round(np.array(Y), 3))

## 2. The QNode on `default.qubit`

This is the inner loop of every PennyLane hybrid job: one fixed circuit structure whose rotation angles the classical optimizer adjusts each step. We encode the input $x$ with `AngleEmbedding`, then apply two trainable `StronglyEntanglingLayers`, and read out $\langle Z_0 \rangle$ -- a value in $[-1, 1]$ that we will push toward $\sin(x)$.

It runs on `qml.device("default.qubit")`, PennyLane's built-in local simulator. No `braket.aws.qubit`, no `device_arn`, no AWS -- this executes entirely on your machine. We keep it at **2 qubits** so it stays fast.

In [ ]:
n_qubits = 2
dev_local = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev_local)
def circuit(params, x):
    # Encode the scalar input on both wires.
    qml.AngleEmbedding(pnp.array([x, x]), wires=range(n_qubits))
    # Trainable variational block.
    qml.StronglyEntanglingLayers(params, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


# Parameter tensor for 2 layers on 2 wires -> shape (2, 2, 3) = 12 angles.
layer_shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=n_qubits)
params = pnp.array(0.3 * np.random.randn(*layer_shape), requires_grad=True)

print("trainable parameter shape:", params.shape, "->", params.size, "angles")
print("prediction at x=0 (untrained):", round(float(circuit(params, X[0])), 4))

## 3. The cost function

The optimizer needs a single scalar to minimize. We use mean-squared error between the circuit's predictions and the targets. Building `preds` with `pnp.stack` (rather than plain `np.array`) keeps the whole computation inside PennyLane's autograd trace, so the optimizer can differentiate through every QNode call to get gradients.

In [ ]:
def cost_fn(params):
    preds = pnp.stack([circuit(params, x) for x in X])
    return pnp.mean((preds - Y) ** 2)


print("initial cost (MSE):", round(float(cost_fn(params)), 5))

## 4. Train with a PennyLane optimizer

Now the loop. `qml.AdamOptimizer` is a pure-Python optimizer that calls `cost_fn`, differentiates it, and returns updated parameters. We log the cost every step into `history` -- the in-notebook stand-in for what `log_metric` streams to CloudWatch inside a real job. Thirty steps is enough to converge on this tiny problem and runs in a couple of seconds locally.

In [ ]:
opt = qml.AdamOptimizer(stepsize=0.2)
steps = 30

history = [float(cost_fn(params))]  # step 0 = starting cost
for step in range(steps):
    params, step_cost = opt.step_and_cost(cost_fn, params)
    history.append(float(step_cost))
    if step % 5 == 0 or step == steps - 1:
        print(f"step {step:2d}  cost = {float(step_cost):.5f}")

print("\nfinal cost:", round(history[-1], 5))

## 5. Plot the convergence

This is the curve a hybrid job would report from inside CloudWatch, except here it ran for real on `default.qubit`. The cost falls steeply over the first dozen steps and settles near zero -- exactly the descent the classical optimizer drives by feeding fresh angles into the fixed circuit each iteration.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(len(history)), history, marker="o", color="#3b82f6")
ax.set_xlabel("optimizer step")
ax.set_ylabel("cost (MSE)")
ax.set_title("Local PennyLane training on default.qubit")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"cost fell from {history[0]:.4f} to {history[-1]:.4f}")

## 6. Retrieve the optimal parameters

After training, `params` holds the optimized angles -- the artifact a job would write to S3 via `save_job_result`. Here we read them straight out of the local loop and spot-check the fitted predictions against the targets.

In [ ]:
optimal_params = np.array(params)
preds = np.array([float(circuit(params, x)) for x in X])

print("optimal_params shape:", optimal_params.shape)
print("\nfitted vs target:")
for xv, pv, yv in zip(np.array(X), preds, np.array(Y)):
    print(f"  x={xv:.2f}  pred={pv:+.3f}  target={yv:+.3f}")

## 7. The job entry-point

Everything above ran locally. To hand this loop to AWS, we move the *same* logic into an entry-point script and change exactly three things:

1. The device becomes `qml.device("braket.aws.qubit", device_arn=os.environ["AMZN_BRAKET_DEVICE_ARN"], ...)`. Braket injects the device ARN into the container's environment, so the script picks up whichever QPU or simulator the job was created against -- and the tasks it submits carry the job token for priority access.
2. Each step calls `log_metric(...)` so the cost streams live to CloudWatch instead of a local list.
3. The final parameters go to `save_job_result(...)`, which persists them to S3 for retrieval.

We define the script as a **string** here. Defining it is free and credential-free; nothing in it runs until a job container executes it. (`pnp.random.randn` replaces the seeded numpy init so the script is self-contained inside the container.)

In [ ]:
ENTRY_POINT = '''import os
import pennylane as qml
from pennylane import numpy as pnp
from braket.jobs.metrics import log_metric
from braket.jobs import save_job_result


def main():
    # Braket injects the device ARN into the container environment.
    device_arn = os.environ["AMZN_BRAKET_DEVICE_ARN"]
    dev = qml.device("braket.aws.qubit", device_arn=device_arn, wires=2)

    @qml.qnode(dev)
    def circuit(params, x):
        qml.AngleEmbedding(pnp.array([x, x]), wires=range(2))
        qml.StronglyEntanglingLayers(params, wires=range(2))
        return qml.expval(qml.PauliZ(0))

    X = pnp.linspace(0.0, 6.283185, 8, requires_grad=False)
    Y = pnp.sin(X)

    def cost_fn(params):
        preds = pnp.stack([circuit(params, x) for x in X])
        return pnp.mean((preds - Y) ** 2)

    shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=2)
    params = pnp.array(0.3 * pnp.random.randn(*shape), requires_grad=True)

    opt = qml.AdamOptimizer(stepsize=0.2)
    for step in range(30):
        params, cost = opt.step_and_cost(cost_fn, params)
        # Streams to CloudWatch; only valid inside a running job.
        log_metric(metric_name="cost", value=float(cost), iteration_number=step)

    # Persists to S3 for retrieval via job.result().
    save_job_result({"optimal_params": params.tolist(), "final_cost": float(cost)})


if __name__ == "__main__":
    main()
'''

print(ENTRY_POINT)
print("entry-point length:", len(ENTRY_POINT), "chars")

## 8. Package and submit (gated)

Writing the script to a local file is harmless. The submission is not -- `AwsQuantumJob.create(...)` spins up a managed instance and starts billing. So it stays behind `if RUN_ON_AWS:`.

**COST NOTE.** A hybrid job adds two charges: the classical instance ($0.10-$3.00/hour while the container runs) plus the usual per-task and per-shot quantum charges -- a job grants priority, not a discount. Per the project rule, prototype on the local simulator first; only flip `RUN_ON_AWS = True` after the loop is validated, and target SV1 before any QPU.

In [ ]:
import pathlib

# Write the entry point to a local source directory (no AWS).
src_dir = pathlib.Path("qml_job_src")
src_dir.mkdir(exist_ok=True)
(src_dir / "qml_train.py").write_text(ENTRY_POINT)
print("wrote entry point ->", src_dir / "qml_train.py")

# Managed simulator ARN (cheapest validation target before any QPU).
SIM_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"

if RUN_ON_AWS:
    # COST NOTE: this starts a billed instance. Caps wall-clock with a stopping_condition (maxRuntimeInSeconds).
    job = AwsQuantumJob.create(
        device=SIM_ARN,
        source_module=str(src_dir),
        entry_point="qml_train:main",
        job_name="pennylane-sine-fit",
        stopping_condition={"maxRuntimeInSeconds": 3600},  # cap wall-clock -> cap cost
        wait_until_complete=False,
    )
    print("submitted job:", job.arn)
else:
    print("RUN_ON_AWS is False -- no job created, no AWS contacted.")
    print("Entry point is staged and ready; flip the flag to submit.")

## 9. Retrieve the optimal parameters from the job (gated)

When a real job finishes, `job.result()` reads back exactly the dict `save_job_result` wrote -- including `optimal_params`. With `RUN_ON_AWS = False` there is no job to query, so we fall back to the parameters our local loop produced in section 6. Same shape, same role: the trained model you carry forward.

In [ ]:
if RUN_ON_AWS:
    result = job.result()  # blocks until the job completes
    recovered = np.array(result["optimal_params"])
    print("recovered optimal_params from S3:", recovered.shape)
    print("final_cost reported by job:", result["final_cost"])
else:
    print("No job to query. Locally trained optimal_params stand in for job.result():")
    print("  shape:", optimal_params.shape)
    print("  local final cost:", round(history[-1], 5))

## Exercises

Four exercises to cement the hybrid-job workflow. Exercises 1-3 train locally on
`default.qubit`, and Exercise 4 edits the job entry point as plain text -- none of
them contacts AWS, so leave `RUN_ON_AWS = False` throughout. Each exercise has
tiered hints -- expand only what you need -- and a check cell that reports when your
attempt holds.

### Exercise 1 — A trainable bias

The circuit reads out $\langle Z_0 \rangle$ in $[-1, 1]$, but nothing lets the
model shift that output up or down. Add a single classical bias term --
`circuit(weights, x) + bias` -- and optimize the weights and the bias *jointly*,
then see whether the fit to $\sin(x)$ improves.

Define `bias_history`: the mean-squared-error cost recorded once per optimizer step
(a starting value plus one per step), and `bias_val`: the trained scalar bias. Run
at least a couple dozen steps so the loop actually converges.

<details><summary>Hint 1 — nudge</summary>

The bias is one extra trainable number sitting outside the circuit. A PennyLane
optimizer can update several trainable arguments at once, as long as your cost
function accepts all of them and every one is marked as a parameter.

</details>
<details><summary>Hint 2 — approach</summary>

Make the bias a differentiable scalar with `pnp.array(0.0, requires_grad=True)`, and
write a cost that takes `(weights, bias)` and stacks `circuit(weights, x) + bias`
over `X`. Call `opt.step_and_cost(cost, weights, bias)` in a loop -- it returns the
updated `(weights, bias)` pair and the step cost -- appending each cost to
`bias_history`. Read `bias_val` off the trained bias at the end.

</details>

In [ ]:
# Exercise 1: Add a trainable bias and optimize it jointly with the weights.
# Define: bias_history -- the MSE cost per optimizer step (start + one per step);
#         bias_val -- the trained scalar bias.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(bias_history) >= 2, "record the cost at each optimizer step"
    assert all(v >= 0.0 for v in bias_history), "an MSE cost can never be negative"
    assert bias_history[-1] < bias_history[0], "training should lower the cost, not raise it"
    assert bias_history[-1] < 0.2, "a jointly trained bias model should drive the MSE near zero"
    assert abs(float(bias_val)) < 2.0, "the bias is a single small offset, not a large number"

### Exercise 2 — Swap the optimizer

`AdamOptimizer` adapts its step size per parameter; plain gradient descent does not.
Retrain the same 2-qubit circuit with `qml.GradientDescentOptimizer(stepsize=0.4)`
alongside `qml.AdamOptimizer(stepsize=0.2)`, starting both from the *same* initial
angles, and compare how their cost curves fall. Plot them on one axis if you want
the picture -- which one wins here may surprise you.

Define `gd_history` and `adam_history`: the per-step MSE cost curves (a starting
value plus one per step) for the gradient-descent and Adam runs respectively, both
over the same number of steps.

<details><summary>Hint 1 — nudge</summary>

Both optimizers expose the same `step_and_cost` interface, so only the constructor
line changes. For the comparison to be about the optimizer and not about luck, what
has to be identical between the two runs before the first step?

</details>
<details><summary>Hint 2 — approach</summary>

Build one initial parameter tensor and reuse a copy of it for each run. For each
optimizer, seed a history list with the starting cost, then loop `step_and_cost` for
the same number of steps, appending each returned cost. `GradientDescentOptimizer`
and `AdamOptimizer` both live in the `qml` namespace.

</details>

In [ ]:
# Exercise 2: Compare GradientDescentOptimizer(0.4) against AdamOptimizer(0.2).
# Define: gd_history, adam_history -- the per-step MSE cost curves (start + one per
#         step) for each optimizer, both from the same initial angles.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert len(gd_history) >= 2 and len(adam_history) >= 2, "record each cost curve step by step"
    assert len(gd_history) == len(adam_history), "run both optimizers for the same number of steps"
    assert all(v >= 0.0 for v in gd_history), "an MSE cost can never be negative"
    assert all(v >= 0.0 for v in adam_history), "an MSE cost can never be negative"
    assert gd_history[-1] < gd_history[0], "gradient descent should lower the cost over training"
    assert adam_history[-1] < adam_history[0], "Adam should lower the cost over training"
    assert min(adam_history) < 0.3, "on this tiny problem the cost should get close to zero"

### Exercise 3 — Deepen the ansatz

The model uses two `StronglyEntanglingLayers`. More layers mean more trainable
angles and more expressivity -- but also a slower, harder optimization. Rebuild the
parameter tensor for **three** layers, retrain on the same target, and see whether
the deeper circuit reaches a lower final cost or merely costs more per step.

Define `deep_params`: the trained three-layer parameter tensor, and
`deep_final_cost`: its final MSE. The existing `circuit` already handles any depth --
the layer count is read from the shape of the parameters you pass it.

<details><summary>Hint 1 — nudge</summary>

`StronglyEntanglingLayers` infers its depth from the first axis of its parameter
tensor, so the *only* thing you change to go deeper is the shape you initialize --
the QNode itself stays exactly as written.

</details>
<details><summary>Hint 2 — approach</summary>

Get the right shape from `qml.StronglyEntanglingLayers.shape(n_layers=3,
n_wires=n_qubits)`, initialize a fresh `requires_grad=True` tensor of small random
angles with it, and run the same Adam training loop as Section 4. After training,
read `deep_params` off the tensor and recompute the MSE for `deep_final_cost`.

</details>

In [ ]:
# Exercise 3: Retrain with a three-layer StronglyEntanglingLayers ansatz.
# Define: deep_params -- the trained 3-layer parameter tensor; deep_final_cost --
#         its final MSE.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    shape = np.asarray(deep_params).shape
    assert len(shape) == 3, "StronglyEntanglingLayers parameters form a (layers, wires, 3) tensor"
    assert shape[0] == 3, "deepen the ansatz to three layers"
    assert shape == (3, n_qubits, 3), "three layers on the same wires means shape (3, n_qubits, 3)"
    assert deep_final_cost >= 0.0, "an MSE cost can never be negative"
    assert deep_final_cost < 0.3, "the retrained three-layer model should still fit sin(x) well"

### Exercise 4 — An early-stopping entry point

A managed job bills for every second the classical instance runs, so a loop that
keeps stepping after the cost has already flattened is wasted money. Take the
`ENTRY_POINT` script from Section 7 and add a stopping condition: once the cost
drops below a small threshold, break out of the training loop early. (You can
re-stage the modified script to the source directory afterward with `write_text`;
the submission itself stays behind `RUN_ON_AWS`.)

Define `stopped_entry_point`: a new copy of the entry-point source string that adds
the early exit while leaving the rest of the job logic -- `main`, `log_metric`,
`save_job_result` -- intact.

<details><summary>Hint 1 — nudge</summary>

The entry point is just a Python program held as a string. Early stopping is an
ordinary `break` guarded by a threshold test on the current cost, placed inside the
training loop -- nothing AWS-specific about it.

</details>
<details><summary>Hint 2 — approach</summary>

Start from the existing `ENTRY_POINT` text and insert two lines into the training
loop, right after the `log_metric(...)` call: an `if float(cost) < <threshold>:`
guard and a `break` beneath it, indented to sit inside the `for` loop. String
methods like `str.replace` are an easy way to splice them in without retyping the
whole script.

</details>

In [ ]:
# Exercise 4: Add an early-stopping break to the job entry-point script.
# Define: stopped_entry_point -- a copy of the ENTRY_POINT source string with a
#         conditional break added inside the training loop.

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert isinstance(stopped_entry_point, str) and stopped_entry_point.strip(), (
        "stopped_entry_point should be the modified script as a string"
    )
    assert "break" in stopped_entry_point, "the loop needs an early exit -- a break statement"
    assert "break" not in ENTRY_POINT, (
        "the stopping condition should be new -- the original script has no break"
    )
    assert "if " in stopped_entry_point, "guard the break with a conditional on the cost"
    for name in ("def main", "log_metric", "save_job_result"):
        assert name in stopped_entry_point, (
            "keep the job plumbing (main, log_metric, save_job_result) intact"
        )

## Summary

- A PennyLane hybrid job is the **same training loop** you ran locally, with the device re-pointed and metrics/results rerouted -- prototype on `default.qubit`, then change three lines.
- The local loop trained a 2-qubit, 12-angle QNode with `AdamOptimizer`, driving the MSE cost from ~0.81 down toward zero -- the convergence curve a job would stream via `log_metric`.
- The entry-point swaps in `qml.device("braket.aws.qubit", device_arn=os.environ["AMZN_BRAKET_DEVICE_ARN"], ...)`, logs each step with `log_metric`, and persists the trained angles with `save_job_result`; `job.result()` reads them back from S3.
- Cost discipline holds: every executed cell ran with no credentials, and `AwsQuantumJob.create` plus `job.result()` stayed behind `RUN_ON_AWS` with a cost note.

**Next:** This is the production capstone of the curriculum. Return to [`../GUIDE.md`](../GUIDE.md) to review the full job lifecycle -- when a job earns its keep, parametric compilation, checkpointing, and cost controls -- or revisit `07-production-patterns.ipynb` for error handling, retries, and pre-submission cost estimation.